# Post-mortem de un pick individual

Analiza un pick pasado: predicción, resultado, features, SHAP, narrativa LLM.

**Uso**: `make notebook NB=05_pick_postmortem BET_ID=12345`

In [ ]:
bet_id = 1

In [ ]:
import asyncio, json
from sqlalchemy import text
from apuestas.db import session_scope

async def load(bid: int) -> dict:
    async with session_scope() as s:
        r = await s.execute(
            text('''
                SELECT b.*, p.probability AS p_model, p.p_lower, p.p_upper,
                       p.features_snapshot, p.shap_top5, p.llm_analysis,
                       m.home_score, m.away_score, m.start_time
                FROM bets b
                LEFT JOIN predictions p ON p.id = b.prediction_id
                LEFT JOIN matches m ON m.id = b.match_id
                WHERE b.id = :bid
            '''),
            {'bid': bid},
        )
        row = r.first()
        return dict(row._mapping) if row else {}

bet = asyncio.run(load(bet_id))
print(f"bet {bet.get('id')} — {bet.get('market')} {bet.get('outcome')} @ {bet.get('odds_placed')} → {bet.get('status')}")
print(f"Score final: {bet.get('home_score')} - {bet.get('away_score')}")
print(f"PNL: {bet.get('pnl_units')} u  CLV: {bet.get('clv')}")

In [ ]:
shap = bet.get('shap_top5') or []
print('Top 5 SHAP features:')
for item in shap:
    print(f"  {item.get('feature')}: {item.get('value'):+.4f}")

In [ ]:
llm = bet.get('llm_analysis') or {}
print(json.dumps(llm, indent=2, ensure_ascii=False)[:2000])

In [ ]:
# Cargar post-mortem si existe
async def load_pm(bid: int) -> dict:
    async with session_scope() as s:
        r = await s.execute(text('SELECT narrative, discrepancy_score FROM post_mortems WHERE bet_id = :b'), {'b': bid})
        row = r.first()
        return dict(row._mapping) if row else {}

pm = asyncio.run(load_pm(bet_id))
if pm:
    print(f"Discrepancy score: {pm.get('discrepancy_score')}")
    print(json.dumps(pm.get('narrative'), indent=2, ensure_ascii=False))
else:
    print('Sin post-mortem aún — correr flow/post_mortem.py')